Обучим на концах CLIP свой классификатор.

In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
from torchvision import datasets
import torchvision
from transformers import CLIPProcessor, CLIPModel

In [3]:
train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomRotation(10),
    torchvision.transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = Imagenette(root = './data', split = 'train', download = True, transform = train_transforms)
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = False, num_workers = 3)

In [4]:
validation_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

validation_dataset = Imagenette(root = './data', split = 'val', download = True, transform = validation_transforms)
validation_loader = DataLoader(validation_dataset, batch_size = 32, shuffle = False, num_workers = 3)

In [8]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
class_names = [labels[0] for labels in train_dataset.classes]
print(class_names)
# Шаманим с моделью

# Заморозка энкодера визуальной составляющей
model.vision_model.requires_grad_(False)

# Хотим сделать классификатор 10 классов => Добавить слой 10 классов
classifier = torch.nn.Linear(512, len(class_names)).to(DEVICE)
model.to(DEVICE)

['tench', 'English springer', 'cassette player', 'chain saw', 'church', 'French horn', 'garbage truck', 'gas pump', 'golf ball', 'parachute']


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [10]:
import torch.optim as optim
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=0.001)

In [18]:
from tqdm import tqdm
# Тренировка
epoch_count = 15
for epoch in range(1, epoch_count + 1):
  print(f"Epoch: {epoch} / {epoch_count}")
  model.eval()
  classifier.train()
  
  curr_loss = 0
  curr_corrects = 0
  for inputs, labels in tqdm(train_loader, "Training"):
    inputs = inputs.to(DEVICE) # картинки
    labels = labels.to(DEVICE)
    
    optimizer.zero_grad()
    
    # Визуальный энкодер заморожен
    with torch.no_grad():
      # Получение эмбендингов картинок
      image_features = model.get_image_features(pixel_values = inputs)
    
    outputs = classifier(image_features)
    _, preds = torch.max(outputs, 1)
    loss = criterion(outputs, labels)
    
    loss.backward()
    optimizer.step()
    
    curr_loss += loss.item() * inputs.size(0)
    curr_corrects += torch.sum(preds == labels.data)
  epoch_loss = curr_loss / len(train_loader.dataset)
  epoch_acc = curr_corrects.double() / len(train_loader.dataset)

  # Валидация
  model.eval()
  classifier.eval()
  val_corrects = 0

  for images, labels in tqdm(validation_loader, "Validation"):
    images = images.to(DEVICE)
    labels = labels.to(DEVICE)
    with torch.no_grad():
      image_features = model.get_image_features(pixel_values=images)
    outputs = classifier(image_features)
    _, preds = torch.max(outputs, 1)
    loss = criterion(outputs, labels)

    val_corrects += torch.sum(preds == labels.data)

  val_acc = val_corrects.double() / len(validation_loader.dataset)
  print(f"Train Loss: {epoch_loss}, Train Acc: {epoch_acc}")
  print(f"Val Acc: {val_acc}")

    

Epoch: 1 / 15


Validation: 100%|██████████| 123/123 [00:23<00:00,  5.15it/s]


Train Loss: 0.42192082407482323, Train Acc: 0.8789734924490442
Val Acc: 0.9765605095541401
Epoch: 2 / 15


Validation: 100%|██████████| 123/123 [00:23<00:00,  5.16it/s]


Train Loss: 0.3396812071784747, Train Acc: 0.9043193579047418
Val Acc: 0.9801273885350318
Epoch: 3 / 15


...

Train Loss: 0.1963336915727864, Train Acc: 0.9387474918153976
Val Acc: 0.9880254777070063
Epoch: 14 / 15


Validation: 100%|██████████| 123/123 [00:23<00:00,  5.14it/s]


Train Loss: 0.19082976948937652, Train Acc: 0.9387474918153976
Val Acc: 0.9880254777070063
Epoch: 15 / 15


Validation: 100%|██████████| 123/123 [00:23<00:00,  5.13it/s]

Train Loss: 0.1853329464122425, Train Acc: 0.9426549794064844
Val Acc: 0.9887898089171974


In [19]:
torch.save(classifier.state_dict(), 'clip_classifier_head.pth')